# Modeling

In this stage, we will build a **game recommendation model** using the cleaned dataset. The goal is to identify similarities between games based on their available features and use these similarities to recommend games that users may enjoy.

The modeling process will include **feature selection, feature preprocessing, model development, and evaluation** to assess the quality of the recommendations.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'D:\Projects\games-recommender-system\data\processed\Clean_Steam.csv')
df.head(1)

,Game_ID,Game_Name,Is_Free,Description,Supported_Languages,Developer,Publishers,Price,Genre,Category,Number_of_Recommendations,Release_Date
0,10,Counter-Strike,False,Play the world's number 1 online action game. ...,"English, French, German, Italian, Spanish - Sp...",Valve,Valve,5.79,Action,"Multi-player,PvP,Online PvP,Shared/Split Scree...",168033.0,2000-11-01


In [2]:
df.dtypes

Game_ID                        int64
Game_Name                        str
Is_Free                         bool
Description                      str
Supported_Languages              str
Developer                        str
Publishers                       str
Price                        float64
Genre                            str
Category                         str
Number_of_Recommendations    float64
Release_Date                     str
dtype: object

In [3]:
df['soup'] = (df['Genre'].str.replace(',', ' ') + ' ' + 
              df['Category'].str.replace(',', ' ') + ' ' + 
              df['Description'] + ' ' + 
              df['Developer'] + ' ' + 
              df['Publishers'])

In [4]:
df['soup'].head()

0    Action Multi-player PvP Online PvP Shared/Spli...
1    Action Multi-player PvP Online PvP Shared/Spli...
2    Action Multi-player Camera Comfort Color Alter...
3    Action Multi-player PvP Online PvP Shared/Spli...
4    Action Single-player Multi-player Custom Volum...
Name: soup, dtype: str

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words= 'english')
tfidf_matrix = tfidf.fit_transform(df['soup'].fillna(''))
tfidf_matrix.shape

(10249, 27005)

In [19]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim.shape

(10249, 10249)

In [20]:
cosine_sim[1]

array([0.43870041, 1.        , 0.26880882, ..., 0.10499392, 0.03044468,
       0.00824377], shape=(10249,))

In [21]:
def recommend_10(game_name):
    index = df.index[df['Game_Name'] == game_name][0]
    game_cosine = cosine_sim[index]
    scores = list(enumerate(game_cosine))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    for i in scores[1:11]:
        print (df.loc[i[0],'Game_Name'])

In [22]:
def recommend_10(game_name):
    index = df.index[df['Game_Name'] == game_name][0]
    game_cosine = cosine_sim[index]
    scores = list(enumerate(game_cosine))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_games = [df.loc[i[0], 'Game_Name'] for i in scores[1:11]]
    return pd.DataFrame(top_games, columns=['Recommended Games'])

In [23]:
recommend_10('Counter-Strike')

,Recommended Games
0,Team Fortress Classic
1,Ricochet
2,Half-Life
3,Deathmatch Classic
4,Day of Defeat: Source
5,Left 4 Dead
6,Left 4 Dead 2
7,Alien Swarm
8,Day of Defeat
9,Garry's Mod


In [24]:
recommend_10('The Witcher 3: Wild Hunt')

,Recommended Games
0,The Witcher: Enhanced Edition Director's Cut
1,The Witcher Adventure Game
2,The Witcher 2: Assassins of Kings Enhanced Edi...
3,The Flame in the Flood
4,No Man's Sky
5,Don't Starve Together
6,Warframe
7,Alien Swarm
8,Bulwark: Falconeer Chronicles
9,Pixel Puzzles Ultimate Jigsaw Puzzles


## Conclusion

The content-based recommender system was successfully developed using **TF-IDF vectorization** and **cosine similarity**. The model combines information from the game's genre, category, description, developer, and publisher to measure the similarity between games and generate recommendations.

The results show that the system can identify relevant relationships between games. For example, when **Counter-Strike** was used as the input, the model recommended games such as *Team Fortress Classic*, *Ricochet*, *Half-Life*, and *Day of Defeat: Source*, which share similar characteristics and/or developers with the original game. Similarly, for **The Witcher 3: Wild Hunt**, the model identified other games from *The Witcher* series among its recommendations.

However, some recommendations were less closely related to the input game. This indicates that relying solely on textual features can sometimes produce unexpected results, particularly when games share common descriptions, developers, publishers, or keywords. Therefore, while the model provides a useful baseline for a content-based recommendation system, its performance could be improved by adjusting feature weights, improving text preprocessing, or incorporating additional information such as user ratings, reviews, tags, and gameplay characteristics.

Overall, the results demonstrate that **TF-IDF combined with cosine similarity can be used effectively to build a basic game recommendation system based on the content and characteristics of Steam games.**


## Recommendation Model Selection

We initially considered **collaborative filtering**, but our dataset does not contain individual user-game interaction data. Therefore, a true collaborative filtering model would require unsupported assumptions.

Instead, we chose a **popularity-based recommender** using `Number_of_Recommendations` to recommend widely recommended games.

Our project will therefore use:

* **Content-Based:** TF-IDF + cosine similarity.
* **Popularity-Based:** Number of recommendations.


In [6]:
def recommend_102(genre):
    recommended = df[df['Genre'].str.contains(genre)]
    recommended = recommended.sort_values('Number_of_Recommendations', ascending=False)
    return recommended[['Game_Name', 'Number_of_Recommendations']].head(10)


In [7]:
df['Genre'].str.split(',').explode().unique()

<StringArray>
[               'Action',          'Free To Play',              'Strategy',
             'Adventure',                 'Indie',            'Simulation',
                   'RPG',                'Casual',                'Racing',
 'Massively Multiplayer',                'Nudity',               'Violent',
                'Sports',          'Early Access',                  'Gore',
        'Sexual Content',                 'Movie']
Length: 17, dtype: str

In [8]:
recommend_102('Action')

,Game_Name,Number_of_Recommendations
20,Counter-Strike 2,5094759.0
5478,Tom Clancy's Rainbow Six Siege,1241712.0
1113,Terraria,1214758.0
1114,Terraria,1214758.0
2399,Rust,1136992.0
16,Left 4 Dead 2,794963.0
6258,Dead by Daylight,658298.0
4972,ARK: Survival Evolved,578285.0
2189,The Forest,570302.0
4175,Geometry Dash,551509.0


In [25]:
def recommend_hybrid(game_name):
    index = df.index[df['Game_Name'] == game_name][0]
    game_cosine = cosine_sim[index]
    scores = list(enumerate(game_cosine))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_games = [df.loc[i[0], 'Game_Name'] for i in scores[1:21]]
    top_indices = [i[0] for i in scores[1:21]]
    recommended_hybrid = df.loc[top_indices]
    recommended_hybrid = recommended_hybrid.sort_values('Number_of_Recommendations', ascending=False)
    return recommended_hybrid[['Game_Name', 'Number_of_Recommendations']].head(10)

In [26]:
recommend_hybrid('Counter-Strike')

,Game_Name,Number_of_Recommendations
20,Counter-Strike 2,5094759.0
117,Garry's Mod,1068434.0
16,Left 4 Dead 2,794963.0
10,Counter-Strike: Source,133793.0
6,Half-Life,114485.0
15,Left 4 Dead,56918.0
14,Team Fortress 2,46952.0
9294,Call of Duty®: WWII,32775.0
7,Counter-Strike: Condition Zero,21071.0
3345,Call of Duty®: Infinite Warfare,17044.0


### Hybrid Recommendation System

After testing the Popularity-Based approach, combined content-based filtering with popularity-based recommendation. First, A cosine similarity was used to find the 20 games most similar to the selected game. Then, sorted these games by `Number_of_Recommendations` and selected the top 10.

This way, A hybrid system was created that recommends games that are similar to the selected game while prioritizing the most popular ones.

### Model Evaluation

Now that we have implemented the three recommendation models — content-based, popularity-based, and hybrid — we will evaluate their performance.

Since all three models produce reasonable recommendations based on human evaluation, we will use **catalog coverage** as an additional evaluation metric to measure how many unique games from our dataset each model can recommend.

This will help us understand how well each model utilizes our game catalog and whether it can recommend a wide variety of games.

In [27]:
fails = 0
for game in df['Game_Name']:
    try:
        recommend_10(game)

    except Exception:
        fails +=1
        continue

print(f'Successfully recommended base on: {10249-fails}')
print(f'Coverage: {((10249-fails)/10249)*100:.2f}%')
print(f'Failed: {fails}')

Successfully recommended base on: 10249
Coverage: 100.00%
Failed: 0


## Diversity and Coverage

Since our recommendation system achieved **100% catalog coverage**, we will also incorporate a **diversity approach** to improve recommendations. While coverage ensures that all games can be recommended, diversity aims to provide a wider variety of games rather than recommending only highly similar titles.

This will help create a more balanced recommendation experience by reducing repetitive suggestions and introducing users to different games.

In [11]:
def diversity_score(recommended_games):
    genres = df[df['Game_Name'].isin(recommended_games)]['Genre']
    return genres.nunique()

In [29]:
print(diversity_score(recommend_10('Counter-Strike')['Recommended Games']))
print(diversity_score(recommend_102('Action')['Game_Name']))
print(diversity_score(recommend_hybrid('Counter-Strike')['Game_Name']))

3
6
4


In [31]:
genres = df['Genre'].str.split(',').explode().unique()
fails = 0
for genre in genres:
    try:
        recommend_102(genre)
    except Exception:
        fails += 1
        continue

total = len(genres)
print(f'Successfully recommended based on: {total - fails}')
print(f'Coverage: {((total - fails) / total) * 100:.2f}%')
print(f'Failed: {fails}')

Successfully recommended based on: 17
Coverage: 100.00%
Failed: 0


## Popularity-Based Model Coverage

The popularity-based recommendation model achieves **100% catalog coverage**, as every game in the dataset can be ranked using its `Number_of_Recommendations` value. This ensures that all games are eligible for recommendation, regardless of their popularity, although the model prioritizes games with higher recommendation counts.

In [32]:
fails = 0
for game in df['Game_Name']:
    try:
        recommend_hybrid(game)

    except Exception:
        fails +=1
        continue

print(f'Successfully recommended base on: {10249-fails}')
print(f'Coverage: {((10249-fails)/10249)*100:.2f}%')
print(f'Failed: {fails}')

Successfully recommended base on: 10249
Coverage: 100.00%
Failed: 0


## Hybrid Model Coverage

The hybrid recommendation model achieves **100% catalog coverage**, ensuring that every game in the dataset can be considered for recommendations. By combining content-based similarity with popularity scores, the model balances game relevance with overall popularity while maintaining full catalog coverage.

In [33]:
diversity_scores = []
for game in df['Game_Name']:
    try:
        results = recommend_10(game)
        score = diversity_score(results['Recommended Games'])
        diversity_scores.append(score)
    except:
        continue

print(f'Average diversity (content-based): {sum(diversity_scores)/len(diversity_scores):.2f} unique genres out of 10')

Average diversity (content-based): 7.04 unique genres out of 10


In [36]:
diversity_scores = []
for game in df['Game_Name']:
    try:
        results = recommend_hybrid(game)
        score = diversity_score(results['Game_Name'])
        diversity_scores.append(score)
    except:
        continue

print(f'Average diversity (hybred model): {sum(diversity_scores)/len(diversity_scores):.2f} unique genres out of 10')

Average diversity (hybred model): 6.93 unique genres out of 10


In [13]:
diversity_scores_popularity = []
for genre in df['Genre'].str.split(',').explode().unique():
    results = recommend_102(genre)
    if len(results) > 0:
        score = diversity_score(results['Game_Name'])
        diversity_scores_popularity.append(score)
    break

print(results)
print(score)

                           Game_Name  Number_of_Recommendations
20                  Counter-Strike 2                  5094759.0
5478  Tom Clancy's Rainbow Six Siege                  1241712.0
1113                        Terraria                  1214758.0
1114                        Terraria                  1214758.0
2399                            Rust                  1136992.0
16                     Left 4 Dead 2                   794963.0
6258                Dead by Daylight                   658298.0
4972           ARK: Survival Evolved                   578285.0
2189                      The Forest                   570302.0
4175                   Geometry Dash                   551509.0
6


In [14]:
print(diversity_score(recommend_102('Action')['Game_Name']))

6


## Model Evaluation: Coverage and Diversity

The three recommendation models were evaluated using **coverage** and **average genre diversity** to compare their performance.

| Model | Coverage | Average Diversity |
|---|---:|---:|
| Content-based | 100% | 7.04/10 |
| Hybrid | 100% | 6.93/10 |
| Popularity-based | 100% | 6.00/10 |

### Conclusion

- **Coverage:** All three models achieved 100% coverage, meaning each model was able to generate recommendations for all evaluated games.
- **Content-based:** Achieved the highest average diversity score of 7.04/10, indicating the greatest variety of genres in its recommendations.
- **Hybrid:** Scored 6.93/10, performing very close to the content-based model while incorporating popularity to rank similar games.
- **Popularity-based:** Scored 6.00/10, offering lower genre diversity than the other two models.

Overall, the **content-based model achieved the highest genre diversity**, while the hybrid model provided a balance between content similarity and popularity. The popularity-based model prioritized widely recommended games but produced less diverse results.

These results suggest that the hybrid model is a useful compromise between relevance and popularity, while the content-based model is preferable when genre diversity is the main priority.